# Flight Delay Analytics — Integrated Exploratory Analysis

**Author:** Daniel Vargas
**Date:** _(fill in)_
**Dataset:** BTS On-Time Performance 2024 + NOAA/IEM ASOS (historical weather)
**Scope:** Top 30 U.S. airports by traffic, full 2024 calendar year

**Business objective:** What factors best predict operational flight delays?

## 0. Import Libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

## 1. Data Acquisition and Integration

### 1.1 BTS Dataset Scope

**Year selected:** 2024 — the most recent complete calendar year available at the time of analysis. 2025 was excluded due to a known data loss period, and 2026 was omitted because it is still in progress with incomplete submissions.

**Airport selection:** The 30 highest-traffic U.S. airports, which concentrate **~65% of all domestic operations** (see top 30 generation cell). Filtering to these airports reduces the dataset from 7 million to 2.5 million rows while retaining the majority of operational significance.

**Volume summary:**
- Raw BTS (all 50 states, 2024): **7,079,061 rows** × 31 columns
- After filtering to top 30 (origin + dest): **2,566,158 rows** (36.2%)
- After merge with weather: **2,566,606 rows** × 47 columns

### 1.2 Integration with NOAA/IEM ASOS

Weather data was downloaded for each of the 30 airports from the Iowa Environmental Mesonet ASOS service. All 30 stations had full 2024 coverage — no station was missing or discarded.

### 1.3 Merge Decisions

**What was discarded:**
- ~4.5M flights where origin or destination was outside the top 30 airports
- ~60 flights that fell on DST transition hours (spring-forward nonexistent or fall-back ambiguous times)
- Temporary processing columns (`dep_local`, `arr_utc`, `dep_hour_utc`, etc.)

**What was approximated:**
- Missing `DepTime`/`ArrTime` (cancelled flights) were replaced with scheduled `CRSDepTime`/`CRSArrTime`
- Weather was merged on the **nearest rounded hour** (±30 min precision) rather than exact minute — reasonable since ASOS observations are hourly
- Arrivals past midnight were detected by comparing against departure time and adding 1 day

**Timezone handling:**
- Each airport was assigned an IANA timezone (e.g. `America/Chicago` for ORD)
- BTS local times were converted to UTC before joining with weather data
- DST transitions: `nonexistent='NaT'` for spring-forward gap, `ambiguous='NaT'` for fall-back overlap; <100 rows affected

In [6]:
# Load BTS data
df_bts = pd.read_csv('../data/ontime_2024_completo.csv')

In [7]:
# Top 30 airports + timezone mapping
top30 = pd.read_csv('../data/processed/top30_airports.csv')

# All top 30 airports are CONUS -> ASOS code = "K" + IATA
top30['icao_asos'] = 'K' + top30['iata']

timezone_map = {
    'ATL': 'America/New_York',    'DFW': 'America/Chicago',
    'DEN': 'America/Denver',      'ORD': 'America/Chicago',
    'CLT': 'America/New_York',    'LAX': 'America/Los_Angeles',
    'PHX': 'America/Phoenix',     'LAS': 'America/Los_Angeles',
    'SEA': 'America/Los_Angeles', 'LGA': 'America/New_York',
    'MCO': 'America/New_York',    'BOS': 'America/New_York',
    'DCA': 'America/New_York',    'SFO': 'America/Los_Angeles',
    'DTW': 'America/New_York',    'EWR': 'America/New_York',
    'MSP': 'America/Chicago',     'JFK': 'America/New_York',
    'IAH': 'America/Chicago',     'SLC': 'America/Denver',
    'MIA': 'America/New_York',    'PHL': 'America/New_York',
    'BNA': 'America/Chicago',     'BWI': 'America/New_York',
    'SAN': 'America/Los_Angeles', 'FLL': 'America/New_York',
    'AUS': 'America/Chicago',     'MDW': 'America/Chicago',
    'TPA': 'America/New_York',    'DAL': 'America/Chicago',
}

top30['timezone'] = top30['iata'].map(timezone_map)
missing_tz = top30[top30['timezone'].isna()]
assert missing_tz.empty, f"Missing timezones for: {missing_tz['iata'].tolist()}"

top30.to_csv('../data/processed/top30_airports.csv', index=False)
top30

,iata,origin_count,dest_count,total_ops,pct_of_total_ops,rank,icao_asos,timezone
0,ATL,341910,341825,683735,4.829277,1,KATL,America/New_York
1,DFW,313582,313568,627150,4.429613,2,KDFW,America/Chicago
2,DEN,308645,308609,617254,4.359717,3,KDEN,America/Denver
3,ORD,280052,280009,560061,3.955758,4,KORD,America/Chicago
4,CLT,217555,217545,435100,3.073148,5,KCLT,America/New_York
5,LAX,194053,194044,388097,2.741162,6,KLAX,America/Los_Angeles
6,PHX,193551,193542,387093,2.734070,7,KPHX,America/Phoenix
7,LAS,189252,189264,378516,2.673490,8,KLAS,America/Los_Angeles
8,SEA,163724,163707,327431,2.312673,9,KSEA,America/Los_Angeles
9,LGA,162432,162470,324902,2.294810,10,KLGA,America/New_York


In [8]:
# NOAA/IEM ASOS weather download for all 30 stations
import httpx
import time
import io
import os
import pandas as pd

top30 = pd.read_csv('../data/processed/top30_airports.csv')

RAW_WEATHER_DIR = '../data/raw/noaa_iem'
os.makedirs(RAW_WEATHER_DIR, exist_ok=True)

SERVICE = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

def download_station(station_id, year=2024):
    params = {
        "station": station_id,
        "data": "all",
        "year1": year, "month1": 1, "day1": 1,
        "year2": year + 1, "month2": 1, "day2": 1,
        "tz": "Etc/UTC",
        "format": "onlycomma",
        "latlon": "no",
        "elev": "no",
        "missing": "M",
        "trace": "T",
        "direct": "no",
        "report_type": 3,
    }
    for attempt in range(6):
        try:
            r = httpx.get(SERVICE, params=params, timeout=300)
            if r.status_code == 200 and len(r.text) > 0:
                return pd.read_csv(io.StringIO(r.text))
        except Exception as e:
            print(f"  {station_id}: attempt {attempt+1} failed ({e})")
        time.sleep(5)
    return None

download_log = []

for _, row in top30.iterrows():
    iata, icao = row['iata'], row['icao_asos']
    fpath = os.path.join(RAW_WEATHER_DIR, f"{iata}.csv")

    if os.path.exists(fpath):
        print(f"{iata}: already exists, skipping")
        continue

    print(f"Downloading {iata} ({icao})...")
    df_weather = download_station(icao)

    if df_weather is None:
        download_log.append({'iata': iata, 'icao': icao, 'status': 'FAILED', 'n_obs': 0})
        print(f"  -> FAILED after 6 attempts")
    else:
        df_weather.to_csv(fpath, index=False)
        download_log.append({'iata': iata, 'icao': icao, 'status': 'OK', 'n_obs': len(df_weather)})
        print(f"  -> OK, {len(df_weather)} observations")

    time.sleep(2)

log_df = pd.DataFrame(download_log)
log_df.to_csv('../data/processed/weather_download_log.csv', index=False)
log_df

ATL: already exists, skipping
DFW: already exists, skipping
DEN: already exists, skipping
ORD: already exists, skipping
CLT: already exists, skipping
LAX: already exists, skipping
PHX: already exists, skipping
LAS: already exists, skipping
SEA: already exists, skipping
LGA: already exists, skipping
MCO: already exists, skipping
BOS: already exists, skipping
DCA: already exists, skipping
SFO: already exists, skipping
DTW: already exists, skipping
EWR: already exists, skipping
MSP: already exists, skipping
JFK: already exists, skipping
IAH: already exists, skipping
SLC: already exists, skipping
MIA: already exists, skipping
PHL: already exists, skipping
BNA: already exists, skipping
BWI: already exists, skipping
SAN: already exists, skipping
FLL: already exists, skipping
AUS: already exists, skipping
MDW: already exists, skipping
TPA: already exists, skipping
DAL: already exists, skipping


""


### Merge: BTS + Weather (code implementation)

See **Section 1.3** above for the full rationale on discarded rows, approximations, and timezone handling. This cell implements the merge described there.

In [10]:
import pandas as pd
import numpy as np
import os

top30_iata = set(top30['iata'])

# --- 1. Filter to top 30 airports (origin AND dest) ---
df = df_bts[
    df_bts['Origin'].isin(top30_iata) & df_bts['Dest'].isin(top30_iata)
].copy()
print(f"Flights in top 30: {len(df):,} ({len(df) / len(df_bts) * 100:.1f}% of total)")

# --- 3. Parse HHMM times to timedelta ---
def hhmm_to_td(val):
    if pd.isna(val):
        return pd.NaT
    val = int(val)
    return pd.Timedelta(hours=val // 100, minutes=val % 100)

# Use actual times, fall back to scheduled
df['DepTime'] = df['DepTime'].fillna(df['CRSDepTime'])
df['ArrTime'] = df['ArrTime'].fillna(df['CRSArrTime'])

df['FlightDate'] = pd.to_datetime(df['FlightDate'])
df['dep_local'] = df['FlightDate'] + df['DepTime'].apply(hhmm_to_td)
df['arr_local'] = df['FlightDate'] + df['ArrTime'].apply(hhmm_to_td)

# --- 4. Handle arrivals crossing midnight ---
cross_midnight = df['arr_local'] < df['dep_local']
df.loc[cross_midnight, 'arr_local'] += pd.Timedelta(days=1)
print(f"Arrivals crossing midnight: {cross_midnight.sum():,}")

# --- 5. Convert local times to UTC (vectorized by timezone group) ---
for tz_name in top30['timezone'].unique():
    airports_in_tz = top30[top30['timezone'] == tz_name]['iata']

    mask_orig = df['Origin'].isin(airports_in_tz)
    df.loc[mask_orig, 'dep_utc'] = (
        df.loc[mask_orig, 'dep_local']
        .dt.tz_localize(tz_name, ambiguous='NaT', nonexistent='NaT')
        .dt.tz_convert('UTC')
    )

    mask_dest = df['Dest'].isin(airports_in_tz)
    df.loc[mask_dest, 'arr_utc'] = (
        df.loc[mask_dest, 'arr_local']
        .dt.tz_localize(tz_name, ambiguous='NaT', nonexistent='NaT')
        .dt.tz_convert('UTC')
    )

# Drop rows with NaT due to DST ambiguity / nonexistent times
print(f"dep_utc NaT: {df['dep_utc'].isna().sum()}, arr_utc NaT: {df['arr_utc'].isna().sum()}")
df = df.dropna(subset=['dep_utc', 'arr_utc'])

# Round to the hour for weather merge (drop tz for merge compatibility)
df['dep_hour_utc'] = df['dep_utc'].dt.round('h').dt.tz_localize(None)
df['arr_hour_utc'] = df['arr_utc'].dt.round('h').dt.tz_localize(None)

# --- 6. Load and prepare weather data ---
weather_dfs = []
for iata in top30_iata:
    fpath = f'../data/raw/noaa_iem/{iata}.csv'
    if not os.path.exists(fpath):
        print(f"WARNING: {fpath} not found, skipping")
        continue
    w = pd.read_csv(fpath, usecols=[
        'station', 'valid', 'tmpf', 'dwpf', 'relh', 'sknt',
        'p01i', 'vsby', 'gust', 'alti', 'feel'
    ])
    w['station'] = iata
    weather_dfs.append(w)

df_weather = pd.concat(weather_dfs, ignore_index=True)
df_weather['valid'] = pd.to_datetime(df_weather['valid'])
df_weather['valid_hour'] = df_weather['valid'].dt.round('h')
print(f"Weather rows loaded: {len(df_weather):,}")

# --- 7. Merge origin weather ---
weather_origin = df_weather.rename(columns={
    'station': 'Origin', 'valid_hour': 'dep_hour_utc',
    'tmpf': 'origin_tmpf', 'dwpf': 'origin_dwpf',
    'relh': 'origin_relh', 'sknt': 'origin_sknt',
    'p01i': 'origin_p01i', 'vsby': 'origin_vsby',
    'gust': 'origin_gust', 'alti': 'origin_alti',
    'feel': 'origin_feel'
}).drop(columns=['valid'])

df = df.merge(weather_origin, on=['Origin', 'dep_hour_utc'], how='left')
print(f"After origin weather merge: {len(df):,} rows")

# --- 8. Merge destination weather ---
weather_dest = df_weather.rename(columns={
    'station': 'Dest', 'valid_hour': 'arr_hour_utc',
    'tmpf': 'dest_tmpf', 'dwpf': 'dest_dwpf',
    'relh': 'dest_relh', 'sknt': 'dest_sknt',
    'p01i': 'dest_p01i', 'vsby': 'dest_vsby',
    'gust': 'dest_gust', 'alti': 'dest_alti',
    'feel': 'dest_feel'
}).drop(columns=['valid'])

df = df.merge(weather_dest, on=['Dest', 'arr_hour_utc'], how='left')
print(f"After dest weather merge: {len(df):,} rows")

# --- 9. Drop temporary columns and save ---
df = df.drop(columns=[
    'DepTime', 'ArrTime', 'dep_local', 'arr_local',
    'dep_utc', 'arr_utc', 'dep_hour_utc', 'arr_hour_utc'
])

out_path = '../data/processed/flights_with_weather.csv'
df.to_csv(out_path, index=False)
print(f"Saved: {out_path} ({len(df):,} rows, {len(df.columns)} columns)")
df.head(3)

Flights in top 30: 2,566,158 (36.2% of total)
Arrivals crossing midnight: 160,972
dep_utc NaT: 10, arr_utc NaT: 59
Weather rows loaded: 263,382
After origin weather merge: 2,566,349 rows
After dest weather merge: 2,566,606 rows
Saved: ../data/processed/flights_with_weather.csv (2,566,606 rows, 47 columns)


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Origin,OriginCityName,OriginState,Dest,...,origin_feel,dest_tmpf,dest_dwpf,dest_relh,dest_sknt,dest_p01i,dest_alti,dest_vsby,dest_gust,dest_feel
0,2024,11,21,4,2024-11-21,OO,LAX,"Los Angeles, CA",CA,SFO,...,48.0,58.0,54.0,86.51,16.0,0.02,30.07,10.0,M,58.0
1,2024,11,21,4,2024-11-21,OO,LAX,"Los Angeles, CA",CA,SFO,...,58.0,60.0,59.0,96.49,10.0,T,30.02,9.0,M,60.0
2,2024,11,21,4,2024-11-21,OO,SFO,"San Francisco, CA",CA,LAX,...,61.0,57.0,51.0,80.31,0.00,0.00,30.1,10.0,M,57.0


## 2. Initial Exploration

_Data types, dimensions, nulls, duplicates._

## 3. Descriptive Statistics

_Central tendency, dispersion, and position — broken down by airline, airport, and time block. Interpret each metric in business context._

## 4. Outlier Detection and Critical Analysis

_Apply IQR or Z-score. Quantify and visualize. Critically evaluate: is the outlier noise or a real signal (storm, mechanical failure)? Decide whether to analyze it separately instead of removing it by default._

## 5. Correlation and Temporal Patterns

_Delay-weather correlation (heatmap). Patterns by time of day, day of week, and season. Identification of "bottleneck" airports._

## 6. Conclusions and Reflection

**Concrete findings:**

**Dataset limitations:**

**Business applications:**

**Ethical considerations:**